### Now with the knowledge graph we built we can now build the recommender model for cv job matching.

#### First Install necessary libraries

In [1]:
import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from gensim.models import Word2Vec

# print("Loading the knowledge graph")
# df_edges = pd.read_csv("kg_edges.csv")

# G = nx.from_pandas_edgelist(df_edges, "source", "target", edge_attr="weight")

# print(f"graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges}, edges")

# print("Beginning of the node2vec training...")


# node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=40, workers=1, quiet=False, temp_folder="temp_walks")
# model = node2vec.fit(window=10, min_count=1, batch_words=4)
# model.save("job_recommender.model")
# print("embedding training finished.")

model = Word2Vec.load("job_recommender.model")


def recommend_jobs(user_skills, top_k=5):

    valid_skills = [s.lower() for s in user_skills if s.lower in model.wv]

    if not valid_skills:
        return "Error: None of the provided skills exist in the Knowledge Graph."

    print(f"Generating recommendations based on: {valid_skills}.")

    user_vector = np.mean([model.wv[s] for s in valid_skills], axis=0)

    similar_nodes = model.wv.most_similar(positive=[user_vector], topn=100)

    recommendations = []
    for node, score in similar_nodes:
        recommendations.append((node,score))
        if len(recommendations) >= top_k:
            break
    return recommendations

df_nodes = pd.read_csv("kg_nodes.csv")
skill_set = set(df_nodes[df_nodes["type"] == "SKILL"]["id"])
title_set = set(df_nodes[df_nodes["type"] == "TITLE"]["id"])

clean_title = title_set - skill_set
print(f"cleaning titles left us with {len(clean_title)} title entries.")
title_nodes = clean_title

def recommend_titles_only(user_skills, top_k=5):
    valid_skills = [s.lower() for s in user_skills if s.lower() in model.wv]
    if not valid_skills: return []

    user_vector = np.mean([model.wv[s] for s in valid_skills], axis=0)

    candidates = model.wv.most_similar(positive=[user_vector], topn=500)

    final_recs = []
    for node, score in candidates:
        if node in title_nodes:
            final_recs.append((node, score))
            if len(final_recs) >= top_k:
                break
    return final_recs

### small test suite for model
sample_profile = ["python", "pandas", "machine learning", "sql"]
recs = recommend_titles_only(sample_profile)

print("Job recommendations")
for job, score in recs:
    job = job.strip("/").title()
    print(f"{job} confidence {score:.2f}")

cleaning titles left us with 340 title entries.
Job recommendations
Machine Learning Engineer confidence 0.85
Python Developer confidence 0.80
Data Analyst confidence 0.79
Data Scientist confidence 0.77
Python Application Developer confidence 0.76


/Users/yusufsurmen/Job-Recommender/Job-Recommender-2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def recommend_skills_for_title(job_title, top_k=10):
    job_title = job_title.lower().strip()
    
    if job_title not in model.wv:
        return [f"Error: '{job_title}' not found in the Knowledge Graph."]
    
    print(f"Generating skill recommendations for: {job_title}")

    job_vector = model.wv[job_title]

    candidates = model.wv.most_similar(positive=[job_vector], topn=100)
    
    recommended_skills = []
    for node, score in candidates:
        if node in skill_set: 
            recommended_skills.append((node, score))
            if len(recommended_skills) >= top_k:
                break
                
    return recommended_skills

target_job = "java developer"
skills = recommend_skills_for_title(target_job)

print(f"\nTop Skills required for '{target_job}':")
for skill, score in skills:
    print(f"- {skill} ({score:.2f})")

Generating skill recommendations for: java developer

Top Skills required for 'java developer':
- hibernate (0.84)
- log4j (0.84)
- j2ee (0.83)
- jdbc (0.83)
- eclipse (0.82)
- jboss eap (0.82)
- wsdl (0.82)
- servlets (0.81)
- jsp (0.81)
- ejb (0.81)


In [ ]:
score = model.wv.similarity("python developer", "java")
print(f"similarity between 'python developer' and 'java' is {score:.2f}")

similarity between 'java developer' and 'python' is 0.48


In [7]:
score = model.wv.similarity("java developer", "photoshop")
print(f"similarity between 'java developer' and 'photoshop' is {score}")

similarity between 'java developer' and 'photoshop' is 0.34558582305908203
